# High-Level BART Tools with bartorch

This notebook demonstrates the high-level Python API provided by `bartorch.tools`.

**bartorch embeds BART directly** — no external `bart` binary is required and no
data is ever written to disk during computation.  All operations run in-memory
via a zero-copy interface between PyTorch tensors and BART's C library.

**Axis convention:** All functions accept C-order Python axis indices (including
negative indices), not BART bitmasks.  For example, `axes=(-1, -2)` selects the
last two axes regardless of tensor rank.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bartorch.tools as bt
from bartorch.core.context import bart_context
from bartorch.utils.cfl import readcfl, writecfl

## `bart_context` — In-Memory Session

The `bart_context` context manager groups a chain of BART operations into a
single in-memory CFL session.  Intermediate tensors never leave C — the
Python ↔ C boundary is crossed only at entry and exit of the `with` block.


In [ ]:
# All ops inside share one in-memory CFL session (pure C hot path)
with bart_context() as ctx:
    ph_ctx = bt.phantom([128, 128])
    ksp_ctx = bt.fft(ph_ctx, axes=(-1, -2))
print(f'Result shape: {ksp_ctx.shape}, dtype: {ksp_ctx.dtype}')


## 1. Phantom Generation

`bt.phantom()` wraps BART's `bart phantom` tool, which synthesises analytically
defined Shepp-Logan phantoms.  The result is a plain `torch.Tensor` with
`complex64` dtype in **C-order** (coils first, last index = readout).  This matches
PyTorch and NumPy conventions and is compatible with every standard PyTorch
operation out of the box.


In [ ]:
# Generate a 256×256 Shepp-Logan phantom
ph = bt.phantom([256, 256])
print(f"Phantom shape: {ph.shape}, dtype: {ph.dtype}")

Because the result is a plain `torch.Tensor`, every standard PyTorch operation
works on it out of the box: `.abs()`, `.real`, `.imag`, `.numpy()`, arithmetic, etc.
The tensor carries `complex64` values — one `complex float` per element — exactly
matching BART's native `complex float*` arrays.  No subclass wrapping, no type
promotion step.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(ph.squeeze().abs().numpy(), cmap='gray')
ax.set_title('Shepp-Logan Phantom')
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Forward FFT (Simulated k-space)

`bt.fft()` calls `bart fft` with a bitmask that selects which dimensions to
transform.  `flags=3` means bits 0 and 1 are set, so the FFT is applied along
both spatial axes simultaneously — the standard 2-D k-space transform for MRI.

The log-magnitude of k-space is shown below; the characteristic bright centre
reflects the concentration of energy at low spatial frequencies.

In [ ]:
kspace = bt.fft(ph, axes=(-1, -2))   # transform dims 0 and 1
print(f"K-space shape: {kspace.shape}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(np.log1p(kspace.squeeze().abs().numpy()), cmap='inferno')
ax.set_title('K-space (log magnitude)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Inverse FFT (Zero-Filled Reconstruction)

`bt.ifft()` is a convenience wrapper around `bart fft -i`.  Applying it to
fully-sampled k-space is an exact round-trip: the reconstructed image should
be numerically identical to the original phantom (up to floating-point rounding).

In [ ]:
reco_zf = bt.ifft(kspace, axes=(-1, -2))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(ph.squeeze().abs().numpy(), cmap='gray')
axes[0].set_title('Original phantom')
axes[0].axis('off')
axes[1].imshow(reco_zf.squeeze().abs().numpy(), cmap='gray')
axes[1].set_title('Zero-filled recon (|IFFT(FFT(x))|)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. Multi-Coil Phantom + ESPIRiT Calibration

Real MRI scanners use phased-array receiver coils.  BART can simulate multi-coil
data via `phantom -k -s <ncoils>`.  Each coil sees a spatially weighted version
of the object; the weights are the *sensitivity maps*.

**ESPIRiT** (`bart ecalib`) estimates those maps from the auto-calibration signal
(ACS) — a small, fully sampled region at the centre of k-space.  The calibration
region size is set by `calib_size`.

In [ ]:
# Generate an 8-coil k-space phantom
kspace_mc = bt.phantom([256, 256], kspace=True, ncoils=8)
print(f"Multi-coil k-space shape: {kspace_mc.shape}")

# Estimate sensitivity maps with ESPIRiT
sens = bt.ecalib(kspace_mc, calib_size=24)
print(f"Sensitivity maps shape: {sens.shape}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    ax = axes[i]
    ax.imshow(sens[..., i, 0].abs().numpy(), cmap='viridis')
    ax.set_title(f'Coil {i+1} sensitivity')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. PICS Reconstruction

**PICS** (Parallel Imaging Compressed Sensing, `bart pics`) is BART's general
iterative reconstruction solver.  It minimises:

$$\hat{x} = \arg\min_x \frac{1}{2}\|A x - y\|_2^2 + \lambda\,\|W x\|_1$$

where $A$ is the SENSE encoding operator (sensitivity maps × FFT), $y$ is the
measured k-space, $W$ is a sparsifying transform (here: wavelets), and
$\lambda$ controls regularisation strength.

The comparison below shows the zero-filled root-sum-of-squares (RSS) reference
alongside the PICS result.  For fully-sampled data the difference is subtle;
the benefit is large for undersampled acquisitions.

In [ ]:
# Reconstruct with L1-Wavelet regularisation
reco = bt.pics(kspace_mc, sens, R='W:7:0:0.005', i=30)
# Multiple regularisers: R=["T:7:0:0.002", "W:7:0:0.005"]
print(f"Reconstruction shape: {reco.shape}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# reference: zero-filled coil combination (RSS)
rss = (kspace_mc.abs()**2).sum(dim=-1).sqrt()
rss_img = bt.ifft(rss, axes=(-1, -2))
axes[0].imshow(rss_img.squeeze().abs().numpy(), cmap='gray')
axes[0].set_title('Zero-filled RSS')
axes[0].axis('off')
axes[1].imshow(reco.squeeze().abs().numpy(), cmap='gray')
axes[1].set_title('PICS L1-Wavelet')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 6. Saving and Loading CFL Files

BART's native array format is **CFL**: a pair of files `<name>.hdr` (ASCII header
with dimensions) and `<name>.cfl` (raw `complex float` binary data).
`bartorch.utils.cfl` provides `writecfl` / `readcfl` that are fully compatible
with BART's own file I/O.  This lets you exchange data with command-line BART
scripts, MATLAB, or other tools.

Note: `readcfl` returns a NumPy array in BART's Fortran-order convention.  To use
the data with bartorch ops, reverse the axis order to match bartorch's C-order
convention (or pass it directly — bartorch will cast and handle the layout
transparently).


In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'phantom_test')
    writecfl(path, ph.numpy())
    loaded = readcfl(path)
    print(f"Written shape: {ph.shape}, Loaded shape: {loaded.shape}")
    print(f"Max abs error: {np.abs(ph.numpy() - loaded).max():.2e}")

## Summary

This notebook walked through the complete bartorch high-level ops interface:

| Step | Function | BART tool |
|------|----------|-----------|
| Phantom generation | `bt.phantom()` | `bart phantom` |
| Forward FFT | `bt.fft(x, flags)` | `bart fft` |
| Inverse FFT | `bt.ifft(x, flags)` | `bart fft -i` |
| ESPIRiT calibration | `bt.ecalib(kspace)` | `bart ecalib` |
| PICS reconstruction | `bt.pics(kspace, sens)` | `bart pics` |
| CFL I/O | `writecfl` / `readcfl` | native BART format |

All functions return plain `torch.Tensor` (complex64) in C-order.
No subclass, no hidden copies.
